In [ ]:
# kaggle_avm_final_stacking.py

import pandas as pd
import numpy as np
import optuna
from sklearn.model_selection import KFold
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PowerTransformer
from sklearn.feature_selection import VarianceThreshold # <-- NEW IMPORT
from sklearn.linear_model import ElasticNet, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

# --- 1. Configuration & Data Loading (Using your reported environment) ---

TRAIN_FILE_PATH = '/kaggle/input/house-prices-advanced-regression-techniques/train.csv'
TEST_FILE_PATH = '/kaggle/input/house-prices-advanced-regression-techniques/test.csv'
TARGET_COLUMN = 'SalePrice'
ID_COLUMN = 'Id'
GRLIVAREA_OUTLIER_THRESHOLD = 4000
N_FOLDS_STACKING = 5
RANDOM_SEED = 42

# --- OPTUNA BEST PARAMETER PLACEHOLDERS (Use your actual values here) ---
OPT_RIDGE_ALPHA = 9.876
OPT_LASSO_ALPHA = 0.000456
OPT_EN_ALPHA = 0.00055
OPT_EN_L1_RATIO = 0.95
OPT_RF_PARAMS = {'n_estimators': 400, 'max_depth': 15, 'min_samples_split': 6, 'min_samples_leaf': 3, 'max_features': 0.75}
OPT_META_GBR_PARAMS = {'n_estimators': 250, 'learning_rate': 0.04, 'max_depth': 3, 'subsample': 0.7}
# -----------------------------------------------------------------

# --- Data Loading (Simulated for this output, will use your Kaggle path live) ---
try:
    train_df_full = pd.read_csv(TRAIN_FILE_PATH)
    test_df = pd.read_csv(TEST_FILE_PATH)
except:
    # Fallback to avoid error in non-Kaggle environment
    from sklearn.datasets import make_regression
    X_sim, y_sim = make_regression(n_samples=1460, n_features=50, n_informative=20, noise=20, random_state=RANDOM_SEED)
    train_df_full = pd.DataFrame(X_sim, columns=[f'F{i}' for i in range(50)])
    train_df_full[TARGET_COLUMN] = np.exp(y_sim / 500 + 11.5)
    train_df_full[ID_COLUMN] = range(1, 1461)
    train_df_full['GrLivArea'] = np.clip(np.abs(X_sim[:, 0] * 50 + 2000), 1000, 5000)
    train_df_full['TotalBsmtSF'] = np.clip(np.abs(X_sim[:, 1] * 30 + 1000), 0, 2000)
    test_df = train_df_full.iloc[:500].copy()


# --- 2. Custom Transformers (Robust to NumPy Array Input) ---

class FeatureEngineer(BaseEstimator, TransformerMixin):
    """Adds TotalSF feature."""
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X_copy = X.copy()
        X_copy['TotalBsmtSF'] = X_copy['TotalBsmtSF'].fillna(0)
        X_copy['TotalSF'] = X_copy['GrLivArea'] + X_copy['TotalBsmtSF']
        return X_copy

class IQROutlierCapper(BaseEstimator, TransformerMixin):
    """Caps numerical outliers using the Interquartile Range (IQR) method (1.5 * IQR)."""
    
    def __init__(self, factor=1.5):
        self.factor = factor
        self.quartiles_ = {}
        self.feature_names_ = None

    def fit(self, X, y=None):
        if isinstance(X, np.ndarray):
            X = pd.DataFrame(X)
        
        self.feature_names_ = list(X.columns)

        for col in X.columns:
            Q1 = X[col].quantile(0.25)
            Q3 = X[col].quantile(0.75)
            IQR = Q3 - Q1
            
            self.quartiles_[col] = {
                'lower_cap': Q1 - (self.factor * IQR),
                'upper_cap': Q3 + (self.factor * IQR)
            }
        return self

    def transform(self, X):
        if isinstance(X, np.ndarray):
             X_capped = pd.DataFrame(X, columns=self.feature_names_ if self.feature_names_ else range(X.shape[1]))
        else:
             X_capped = X.copy()
        
        for col in X_capped.columns:
            fences = self.quartiles_.get(col)
            if fences:
                lower_cap = fences['lower_cap']
                upper_cap = fences['upper_cap']
                X_capped[col] = np.clip(X_capped[col], a_min=lower_cap, a_max=upper_cap)
            
        return X_capped.values

# --- 3. Initial Outlier Handling and Data Prep ---

train_df_full_fe = FeatureEngineer().transform(train_df_full)
test_df_fe = FeatureEngineer().transform(test_df)

train_df_cleaned = train_df_full_fe.copy()
train_df_cleaned = train_df_cleaned[train_df_cleaned['GrLivArea'] < GRLIVAREA_OUTLIER_THRESHOLD].reset_index(drop=True)
top_2_outliers = train_df_cleaned[TARGET_COLUMN].nlargest(2).index
train_df_cleaned = train_df_cleaned.drop(top_2_outliers, axis=0).reset_index(drop=True)

print(f"\n--- Data Cleaning Complete. Total records: {train_df_cleaned.shape[0]} ---")

X_full = train_df_cleaned.drop([TARGET_COLUMN, ID_COLUMN], axis=1)
y_log_full = np.log1p(train_df_cleaned[TARGET_COLUMN])
X_test = test_df_fe.drop(ID_COLUMN, axis=1)
test_ids = test_df_fe[ID_COLUMN]

numeric_features = X_full.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_full.select_dtypes(include='object').columns.tolist()

# --- 4. Preprocessing Pipeline Definition (FIXED) ---

def get_preprocessor(numeric_features, categorical_features):
    """Creates the complete ColumnTransformer preprocessor with IQR capping and VarianceThreshold."""
    
    numeric_pipeline = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),  
        ('iqr_capper', IQROutlierCapper(factor=1.5)), 
        
        # <<< FIX: REMOVE CONSTANT FEATURES THAT BREAK POWERTRANSFORM >>>
        ('variance_killer', VarianceThreshold(threshold=0.0)), 
        
        ('power_transform', PowerTransformer(method='yeo-johnson')), 
        ('scaler', StandardScaler())  
    ])

    categorical_pipeline = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_pipeline, numeric_features),
            ('cat', categorical_pipeline, categorical_features)
        ],
        remainder='passthrough',
        n_jobs=-1
    )
    return preprocessor

# --- 5. Final Stacking Ensemble (Auto Blend) ---

print("\n--- 5. Stacking Ensemble (Auto Blend) with 4 Base Models ---")

# Define the 4 final, best-tuned base models
base_model_ridge = Ridge(alpha=OPT_RIDGE_ALPHA, random_state=RANDOM_SEED)
base_model_lasso = Lasso(alpha=OPT_LASSO_ALPHA, max_iter=5000, random_state=RANDOM_SEED)
base_model_en = ElasticNet(alpha=OPT_EN_ALPHA, l1_ratio=OPT_EN_L1_RATIO, max_iter=5000, random_state=RANDOM_SEED)
base_model_rf = RandomForestRegressor(random_state=RANDOM_SEED, **OPT_RF_PARAMS)

base_models = [
    ('Ridge', base_model_ridge),
    ('Lasso', base_model_lasso),
    ('ElasticNet', base_model_en),
    ('RandomForest', base_model_rf)
]

N_TRAIN = X_full.shape[0]
N_TEST = X_test.shape[0]
oof_predictions = np.zeros((N_TRAIN, len(base_models)))
test_predictions = np.zeros((N_TEST, len(base_models)))
kf = KFold(n_splits=N_FOLDS_STACKING, shuffle=True, random_state=RANDOM_SEED)

# --- A. Generate Out-of-Fold (OOF) and Test Predictions ---

for i, (name, model) in enumerate(base_models):
    print(f" > Generating OOF and Test predictions for: {name}")

    full_pipeline = Pipeline(steps=[
        ('feature_engineer', FeatureEngineer()), 
        ('preprocessor', get_preprocessor(numeric_features, categorical_features)),
        ('regressor', model)
    ])

    test_fold_predictions = np.zeros(N_TEST)

    for fold, (train_idx, val_idx) in enumerate(kf.split(X_full, y_log_full)):
        X_train_fold, X_val_fold = X_full.iloc[train_idx], X_full.iloc[val_idx]
        y_train_fold, y_val_fold = y_log_full.iloc[train_idx], y_log_full.iloc[val_idx]

        full_pipeline.fit(X_train_fold, y_train_fold) 
        oof_predictions[val_idx, i] = full_pipeline.predict(X_val_fold)
        test_fold_predictions += full_pipeline.predict(X_test) / N_FOLDS_STACKING

    test_predictions[:, i] = test_fold_predictions


# --- B. Train the Meta-Model (Blender) ---

print("\n > Training Meta-Model (Gradient Boosting Regressor) on OOF predictions...")
meta_model = GradientBoostingRegressor(loss='squared_error', random_state=RANDOM_SEED, **OPT_META_GBR_PARAMS)
meta_model.fit(oof_predictions, y_log_full)


# --- C. Final Prediction and Submission ---

y_pred_log_blended_test = meta_model.predict(test_predictions)
y_pred_dollar_blended_test = np.expm1(y_pred_log_blended_test)

SUBMISSION_FILE_NAME = 'submission_final_tuned_stacking.csv'
submission_df = pd.DataFrame({
    'Id': test_ids,
    TARGET_COLUMN: y_pred_dollar_blended_test
})

print(f"\n--- Final Stacking Submission Created: {SUBMISSION_FILE_NAME} ---")
print(f"Predictions blended using a tuned Gradient Boosting Regressor meta-model.")
print(submission_df.head())